# Using Population Data from epydemix_data for Epidemiological Modeling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/epistorm/epydemix/blob/main/tutorials/02_Modeling_with_Population_Data.ipynb)

In this tutorial, we will explore how to use population data from the epydemix_data package to model the spread of diseases. We'll take a practical approach by demonstrating how to load population data, integrate it with an epidemiological model, and run simulations using the ***epydemix*** package.

If you are running this tutorial on Google Colab, you can install the needed packages by running the following cell:



In [ ]:
import sys,os,subprocess
if "google.colab" in sys.modules or os.getenv("COLAB_RELEASE_TAG"):
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","https://raw.githubusercontent.com/epistorm/epydemix/refs/heads/main/tutorials/colab_requirements.txt"])

## Working with Population Objects

By default, when an ```EpiModel``` instance is created without specific parameters, ***epydemix*** will create a single population with 100000 individuals homogeneously mixed. If you are interested in running your model on demographic data from a specific geography in the world, you can do so by easily import the data contained in the epydemix_data module. 

**Epydemix** supports real-world population data and synthetic contact matrices for epidemic simulation for more than $400$ regions worldwide. A comprehensive list of supported geographies can be found in the [locations.csv](https://github.com/epistorm/epydemix-data/blob/main/locations.csv) file, while additional information on population data including sources can be found in the [epydemix_data README](https://github.com/epistorm/epydemix-data).

The **Epydemix** package provides flexibility in loading demographic data and contact matrices for various regions. The `epydemix.model.load_epydemix_population` function allows you to load this data either via **online import** (fetching from a remote source) or **offline import** (loading from a local directory).

- **Online Import**: If `path_to_data` is not provided, **Epydemix** will attempt to import the data directly from GitHub.
- **Offline Import**: If `path_to_data` is provided, **Epydemix** will attempt to load the data from a local directory. For this option, users must first download the corresponding data folder.

Let's see a practical example, importing data for Kenya:

In [ ]:
from epydemix.population import load_epydemix_population

# Load the population of Kenya from GitHub. For offline use, download the population data from
# https://github.com/epistorm/epydemix_data and provide the local path to 
# the epydemix_data folder using the argument path_to_data

population = load_epydemix_population("Japan")

print(population)

We see that the ```Population``` object has the following key attributes:

- Population Distribution: this represents the number of individuals in each demographic group (e.g., age groups). By default, population are imported divided into 5 age groups (0-4, 5-19, 20-49, 50-64, 65+) but the granularity can be changed by specifying the ```age_group_mapping``` argument.

- Contact Matrices: these represent contact rates in different contexts (e.g., school, home). More in detail, each element $i, j$ of the contact matrix represents the average rate of contact that an individual in demographic group $i$ has with individuals in demographic group $j$ in a unit of time (e.g., one day).


We can explore population distribution and contact matrices using the ***epydemix*** visualization module:

In [ ]:
from epydemix.visualization import plot_contact_matrix, plot_population
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=2, ncols=2, dpi=300)
plot_contact_matrix(population, "school", ax=axes[0,0], fontsize=7, show_values=True)
plot_contact_matrix(population, "work", ax=axes[0,1], fontsize=7, show_values=True)
plot_contact_matrix(population, "home", ax=axes[1,0], fontsize=7, show_values=True)
plot_contact_matrix(population, "community", ax=axes[1,1], fontsize=7, show_values=True)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(ncols=2, dpi=300, figsize=(10, 5))
plot_population(population, ax=axes[0], title="Population Distribution (absolute numbers)")
plot_population(population, ax=axes[1], title="Population Distribution (percentages)", show_perc=True)

Alternatively, you can create your own population using your data by directly creating an instance of the ```Population``` class:

In [ ]:
from epydemix.population import Population

my_population = Population(name="My Population")    
my_population.add_population(Nk=[100, 100], Nk_names=["A", "B"])
my_population.add_contact_matrix(contact_matrix=[[0.2, 0.3], [0.3, 0.2]], layer_name="all")

print(my_population)

## Simulating Epidemic Models on Real-World Populations

Now that we have imported or created our Population object, we can simulate an epidemic model considering this population instead of the default one.

In [ ]:
from epydemix import load_predefined_model

# create a simple SIR model
model = load_predefined_model("SIR", transmission_rate=0.04)

# set the population (alternatively you can import it using model.import_epydemix_population('Kenya'))    
model.set_population(population)

print(model)

In [ ]:
from epydemix.visualization import plot_quantiles

# simulate 
results = model.run_simulations(start_date="2019-12-01", 
                                end_date="2020-04-01", 
                                percentage_in_agents=10 / model.population.Nk.sum(),
                                Nsim=100)

# plot results
df_quantiles_comps = results.get_quantiles_compartments()
ax = plot_quantiles(df_quantiles_comps, columns=["Infected_total", "Susceptible_total", "Recovered_total"], legend_loc="center right")

We can also inspect the evolution of compartments in specific age groups:

In [ ]:
ax = plot_quantiles(df_quantiles_comps, columns=["Infected_0-4", "Infected_5-19", "Infected_20-49", "Infected_50-64", "Infected_65+"], legend_loc="center right")

### Setting a Different Age Group Granularity

As detailed above, by default population are imported divided into 5 age groups (0-4, 5-19, 20-49, 50-64, 65+) but the granularity can be changed by specifying the ```age_group_mapping``` argument.

It is important to note that the maximum granularity depends on the contact matrix source considered, more in detail: 

- Mistry 2021 matrices have a maximum granularity of 85 age groups, ranging 0, 1, ..., 83, 84+
- Prem 2017 and Prem 2021 have a maximum granularity of 16 age groups, ranging 0-4, 5-9, ..., 70-74, 75+

Let's see an example where we import a population setting a different age group mapping:

In [ ]:
population = load_epydemix_population("Kenya", 
                                      age_group_mapping={"0-9": ["0-4", "5-9"],
                                                         "10-19": ["10-14", "15-19"],
                                                         "20-29": ["20-24", "25-29"],
                                                         "30-39": ["30-34", "35-39"],
                                                         "40-49": ["40-44", "45-49"],
                                                         "50-59": ["50-54", "55-59"],
                                                         "60-69": ["60-64", "65-69"],
                                                         "70+": ["70-74", "75+"]},
                                      contacts_source="prem_2021")

print(population)